# Module 2: Real data with MCP

In Module 1 your agent could reason and use tools, but it knew nothing about your
account. In this module you will give it real data. You will talk to a Model Context
Protocol (MCP) server directly to see the protocol, watch an agent guess without it,
then connect the server to a [Strands Agent](https://strandsagents.com) so it reads your
actual Akamai Cloud resources. The reads come from `akamai-cloud-mcp`, a read-only server
for Akamai Cloud (Linode).

## Learning objectives
- Understand what an MCP server is and how an agent reaches it over a transport
- Call an MCP server directly: `initialize`, `list_tools`, `call_tool`
- See an agent guess about your account when it has no tools for it
- Connect the MCP server to a Strands agent so it reads live data
- Scope the tool set so a small model is not overwhelmed

## Prerequisites
- Finished Module 1
- A read-only [Linode API token](https://techdocs.akamai.com/cloud-computing/docs/manage-personal-access-tokens) in `LINODE_TOKEN`
- The [akamai-cloud-mcp](https://github.com/akamai-developers/akamai-cloud-mcp) server
- About 25 minutes

References: [Model Context Protocol](https://modelcontextprotocol.io) · [Strands MCP tools](https://strandsagents.com) · [akamai-cloud-mcp](https://pypi.org/project/akamai-cloud-mcp/) · [Linode API](https://techdocs.akamai.com/linode-api/reference/api)

## Model Context Protocol (MCP) design basics

An MCP server exposes tools (and resources) over a standard protocol, so any agent can
use them without custom configuration. Your agent is the MCP *client*. It launches or connects to
the *server*, asks it to `list_tools`, and calls them with `call_tool`. The server in this lab,
`akamai-cloud-mcp`, turns the Linode API into read-only tools and scrubs secrets before
they ever reach the model.

![The agent is an MCP client that lists and calls tools on the akamai-cloud-mcp server, which reads the Linode API](../images/02_mcp_architecture.png)

## 1. Setup

The MCP Python SDK ships as `mcp`. Strands and its tools are already installed from
Module 1; we reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q mcp python-dotenv "strands-agents[openai]" strands-agents-tools

## 2. Configure the model, token, and server

`LINODE_TOKEN` is read from your `.env`. `AKAMAI_MCP_DIR` points at a local checkout of
the server; leave it empty to run the published package with `uvx`. 

The model values are the same vLLM endpoint from Module 1.

In [ ]:
import os
from dotenv import load_dotenv
from mcp import StdioServerParameters

load_dotenv()

BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
MODEL_ID = os.getenv("VLLM_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
API_KEY = os.getenv("VLLM_API_KEY", "placeholder")

LINODE_TOKEN = os.getenv("LINODE_TOKEN", "")
MCP_DIR = os.getenv("AKAMAI_MCP_DIR", "")  # local checkout; empty -> use the published package

def mcp_server(domains="regions,lke,compute", max_results=10):
    """Describe how to launch the akamai-cloud-mcp server over stdio."""
    scope = ["--domains", domains, "--max-results", str(max_results)]
    if MCP_DIR:
        # Run the server from a local clone of the repository
        command, args = "uv", ["run", "--directory", MCP_DIR, "akamai-cloud-mcp", *scope]
    else:
        # Run the published package from pypi
        command, args = "uvx", ["akamai-cloud-mcp", *scope]
    # The token goes to the server through the environment, never to the model.
    return StdioServerParameters(command=command, args=args, env={**os.environ, "LINODE_TOKEN": LINODE_TOKEN})

print("Token set:", bool(LINODE_TOKEN))
print("Local server dir:", MCP_DIR or "(using published package via uvx)")

## 3. Invoke the MCP server directly

Before using a framework with mcp, we can invoke the protocol directly to show that it is not a black box. You
launch a mcp server locally over stdio, `initialize` the session, ask for its tools with `list_tools`, then run one with `call_tool`. We then print the raw results.

(The cell uses `await` directly. Jupyter runs an event loop, so top-level `await` works in
a cell.)

In [ ]:
from mcp import ClientSession
from mcp.client.stdio import stdio_client

server = mcp_server(domains="regions", max_results=5)

async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        # Handshake: the client and server agree on protocol version and capabilities
        await session.initialize()

        # Ask the server what tools it exposes
        listed = await session.list_tools()
        print(f"The server exposes {len(listed.tools)} tool(s):")
        for t in listed.tools:
            print(f" - {t.name}: {t.description[:70]}")

        # Look at one tool's input schema, the contract the model would use
        first = listed.tools[0]
        print(f"\nInput schema for {first.name}:")
        print(first.inputSchema)

        # Call a tool and print the raw result the server returned
        result = await session.call_tool("linode_list_regions", {})
        print("\nRaw call_tool result (first 400 chars):")
        print(result.content[0].text[:400])

That is the MCP protocol: connect, list, call. The server did the Linode API request, shaped the response, and scrubbed anything sensitive. Your client never saw the token or any secret field.

## 4. Without your data, the model can only guess

In this example we ask the model about our account before it has any account tools. If you recall from Module 1, vLLM rejects a tool-less agent, and an agent with no account tools could not
answer this accurately. When we ask the raw model directly it has no way to know your account, so it uses its training data to make a prediction based on probabilities (guess) or the model admits it cannot answer. 

We also build the model object here for the agent in
the next sections.

In [ ]:
import httpx
from strands import Agent
from strands.models.openai import OpenAIModel

# The model object our agent will use in the next sections
model = OpenAIModel(
    client_args={"api_key": API_KEY, "base_url": BASE_URL},
    model_id=MODEL_ID,
    params={"temperature": 0.2},
)

SYSTEM_PROMPT = (
    "You are an Akamai Cloud Solutions Architect. Prefer calling a tool over guessing "
    "whenever the answer depends on real account data."
)

# Ask the raw model, with no tools, about your account
resp = httpx.post(
    f"{BASE_URL}/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How many LKE clusters do I have, and in which regions?"},
        ],
        "temperature": 0.2,
    },
    timeout=60,
)

print(resp.json()["choices"][0]["message"]["content"])

## 5. Connect the MCP server to your agent

Now we will give the MCP tools to the agent. The Strands' `MCPClient` launches the server and turns the tools into agent tools. `list_tools_sync()` is the same `list_tools` you called by hand in section 3, and when the agent calls one of the tools, Strands runs the same `call_tool` for you.

One rule: the `MCPClient` owns the server process, so the agent must run while the client is open. Here we handle that with the `with` block.

In [ ]:
from strands.tools.mcp import MCPClient
from mcp.client.stdio import stdio_client

# Build the client. The lambda tells it how to start the server (the same params as before).
mcp_client = MCPClient(lambda: stdio_client(mcp_server(domains="regions,lke,compute", max_results=10)))

with mcp_client:
    # list_tools_sync is the framework doing the list_tools you did by hand
    account_tools = mcp_client.list_tools_sync()
    print(f"Loaded {len(account_tools)} account tools, for example:",
          ", ".join(t.tool_name for t in account_tools[:5]), "...\n")

    agent = Agent(model=model, tools=account_tools, system_prompt=SYSTEM_PROMPT)

    # The agent must run inside the with block, while the server is alive
    _ = agent("How many LKE clusters do I have, and in which regions? Use your tools.")

The agent provided the response using your real Akamai Cloud account. The agent listed the tools, picked
`linode_list_lke_clusters`, called it through MCP, and read the result using the same ReAct loop from Module 1. All with the tools that were provided by the MCP server.

## 6. Scope the tools for a small model

The `akamai-cloud-mcp` can expose dozens of tools. A small model makes better decisions when it only has to choose from a short, relevant list. In this example we scope the tools given to the agent with `--domains` and cap rows with `--max-results`. This loads only the tools for the tasks the agent needs.

Scoping tools or using progressive disclosure with MCP helps not only to improve the agent's performance but it also prevents token bloat which reduces costs.

In [ ]:
# Pricing and GPU questions only need the pricing domain
with MCPClient(lambda: stdio_client(mcp_server(domains="pricing", max_results=5))) as priced:
    pricing_tools = priced.list_tools_sync()
    print("Pricing tools:", ", ".join(t.tool_name for t in pricing_tools))

    agent = Agent(model=model, tools=pricing_tools, system_prompt=SYSTEM_PROMPT)
    _ = agent("What GPU plans are available in us-sea, and what do they cost per month? Use your tools.")

## 7. Things to know about the Akamai Cloud MCP server design

- **Read-only by design:** This server ships no write or mutating tools. Reads are safe to hand to an agent. Changes are a different problem, with their own guardrail. We will cover more on this in Module 3.
- **Secrets never reach the model.** The server scrubs tokens, access keys, kubeconfigs,
  and payment fields out of every response. The token lives in the server's environment.
- **Tool names are prefixed.** Account tools are named `linode_list_lke_clusters`,
  `linode_get_pricing`, and so on. If your code expects a tool name, match the prefix.
- **Two transports.** Here we used `stdio`, where the client launches the server as a local
  process. For a hosted deployment the same server runs over streamable HTTP behind a bearer
  token. The agent code barely changes.

## Try it yourself

Make the agent answer a question about a part of your account you have not touched yet.

1. **Pick a new domain.** Change `your_domains` below to one you have not used: `networking`, `object_storage`, or `account`.
2. **Ask a question that needs it.** For example:
   - `networking` &rarr; "List my VPCs and NodeBalancers."
   - `object_storage` &rarr; "What Object Storage buckets do I have, and in which regions?"
   - `account` &rarr; "What is my account balance and recent activity?"

**Stretch:** scope to `compute,pricing` and ask the agent to estimate your current monthly compute cost. It has to list your instances, look up each plan's price, and add them up: several tool calls in one loop.

In [ ]:
# Change these two lines, then run the cell.
your_domains = "networking"      # try: networking, object_storage, account, or compute,pricing
your_question = "List my VPCs and NodeBalancers."

with MCPClient(lambda: stdio_client(mcp_server(domains=your_domains, max_results=10))) as client:
    tools = client.list_tools_sync()
    print("Tools loaded:", ", ".join(t.tool_name for t in tools), "\n")
    agent = Agent(model=model, tools=tools, system_prompt=SYSTEM_PROMPT)
    _ = agent(your_question)

## Summary

- An MCP server exposes tools over a protocol: `initialize`, `list_tools`, `call_tool`.
- Without account tools the agent guesses. With them it reads real data.
- Strands' `MCPClient` does the `list_tools` and `call_tool` you did by hand, and the
  server keeps every secret away from the model.

## Next

**Module 3: Guardrails.** Reads are safe, but changes are not. Next you will let the agent
make changes only after a human approves the exact action, with a gate the model cannot
bypass.